In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
import os

print("Current Working Directory:")
print(os.getcwd())

Current Working Directory:
d:\MedTrack_DV\notebooks


In [4]:
import pandas as pd
import os

BASE_DIR = os.path.dirname(os.getcwd())

DATA_PATH = os.path.join(
    BASE_DIR,
    "data",
    "cleaned",
    "hospital_cleaned.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (45000, 58)


,admission_id,admission_date,discharge_date,admission_type,admission_status,patient_id,department_id,ward_id,bed_id,disease_id,...,admission_year,admission_month,admission_month_number,admission_year_month,admission_quarter,emergency_flag,stay_category,department_revenue,disease_load,bed_occupancy_flag
0,10703,2020-01-01,2020-01-04,Emergency,Discharged,24223,4,19,276,17,...,2020,January,1,2020-01,Q1,1,Short Stay,42384,2259,1
1,1044,2020-01-01,2020-01-06,Elective,Discharged,19446,5,23,349,5,...,2020,January,1,2020-01,Q1,0,Medium Stay,36438,2275,1
2,9446,2020-01-01,2020-01-02,Emergency,Discharged,17151,5,21,301,5,...,2020,January,1,2020-01,Q1,1,Short Stay,33337,2275,1
3,33351,2020-01-01,2020-01-02,Emergency,Discharged,9933,4,17,258,1,...,2020,January,1,2020-01,Q1,1,Short Stay,16881,2282,1
4,21625,2020-01-01,2020-01-06,Elective,Discharged,8146,4,18,270,8,...,2020,January,1,2020-01,Q1,0,Medium Stay,44904,2251,1


In [5]:
df.columns.tolist()

['admission_id',
 'admission_date',
 'discharge_date',
 'admission_type',
 'admission_status',
 'patient_id',
 'department_id',
 'ward_id',
 'bed_id',
 'disease_id',
 'gender',
 'date_of_birth',
 'blood_group',
 'city',
 'contact_number',
 'department_name',
 'department_type',
 'floor_number',
 'status',
 'ward_name',
 'ward_type',
 'total_beds',
 'bed_number',
 'bed_status',
 'disease_name',
 'disease_category',
 'bill_id',
 'bill_date',
 'total_amount',
 'insurance_covered_amount',
 'patient_payable_amount',
 'payment_status',
 'payment_mode',
 'patient_insurance_id',
 'policy_number',
 'coverage_percentage',
 'policy_start_date',
 'policy_end_date',
 'insurance_provider_id',
 'provider_name',
 'provider_type',
 'contact_details',
 'coverage_limit',
 'length_of_stay',
 'patient_age',
 'age_group',
 'insurance_status',
 'revenue_category',
 'admission_year',
 'admission_month',
 'admission_month_number',
 'admission_year_month',
 'admission_quarter',
 'emergency_flag',
 'stay_categor

In [6]:
total_admissions = df["admission_id"].count()

print("Total Admissions:", total_admissions)

Total Admissions: 45000


In [7]:
occupied_beds = df["bed_occupancy_flag"].sum()

total_records = len(df)

occupancy_rate = (occupied_beds / total_records) * 100

print(f"Occupancy Rate: {occupancy_rate:.2f}%")

Occupancy Rate: 100.00%


In [8]:
average_los = df["length_of_stay"].mean()

print(f"Average Length of Stay: {average_los:.2f} days")

Average Length of Stay: 5.16 days


In [9]:
patient_counts = df.groupby("patient_id").size()

readmitted_patients = (patient_counts > 1).sum()

total_patients = df["patient_id"].nunique()

readmission_rate = (readmitted_patients / total_patients) * 100

print(f"Readmission Rate: {readmission_rate:.2f}%")

Readmission Rate: 56.92%


In [10]:
bed_utilization_rate = df["bed_occupancy_flag"].mean() * 100

print(f"Bed Utilization Rate: {bed_utilization_rate:.2f}%")

Bed Utilization Rate: 100.00%


In [11]:
department_efficiency = (
    df.groupby("department_name")
      .agg(
          Total_Admissions=("admission_id", "count"),
          Average_Length_of_Stay=("length_of_stay", "mean"),
          Total_Revenue=("department_revenue", "sum")
      )
)

department_efficiency["Efficiency_Score"] = (
    department_efficiency["Total_Revenue"] /
    department_efficiency["Average_Length_of_Stay"]
)

department_efficiency.sort_values(
    "Efficiency_Score",
    ascending=False
)

,Total_Admissions,Average_Length_of_Stay,Total_Revenue,Efficiency_Score
department_name,,,,
Surgery,10126,4.674798,377216234,8.069146e+07
Emergency,8777,4.692492,329738229,7.026933e+07
Pediatrics,8438,4.691752,315714822,6.729146e+07
Internal Medicine,7695,4.656140,289566059,6.219015e+07
Orthopedics,5924,4.677920,222585716,4.758219e+07
ICU,4040,9.980693,149425049,1.497141e+07


In [12]:
df["Total_Admissions"] = total_admissions
df["Occupancy_Rate"] = round(occupancy_rate, 2)
df["Average_Length_of_Stay"] = round(average_los, 2)
df["Readmission_Rate"] = round(readmission_rate, 2)
df["Bed_Utilization_Rate"] = round(bed_utilization_rate, 2)

In [13]:
OUTPUT_PATH = os.path.join(
    BASE_DIR,
    "data",
    "processed",
    "hospital_final_dataset.xlsx"
)

df.to_excel(OUTPUT_PATH, index=False)

print("hospital_final_dataset.xlsx created successfully!")

hospital_final_dataset.xlsx created successfully!


In [14]:
# Merge department efficiency score back into the dataset

df = df.merge(
    department_efficiency[["Efficiency_Score"]],
    left_on="department_name",
    right_index=True,
    how="left"
)

# Rename the column
df.rename(
    columns={"Efficiency_Score": "Department_Efficiency_Score"},
    inplace=True
)

print(df[["department_name", "Department_Efficiency_Score"]].head())

  department_name  Department_Efficiency_Score
0      Pediatrics                 6.729146e+07
1     Orthopedics                 4.758219e+07
2     Orthopedics                 4.758219e+07
3      Pediatrics                 6.729146e+07
4      Pediatrics                 6.729146e+07


In [15]:
OUTPUT_PATH = os.path.join(
    BASE_DIR,
    "data",
    "processed",
    "hospital_final_dataset.xlsx"
)

df.to_excel(OUTPUT_PATH, index=False)

print("Updated hospital_final_dataset.xlsx saved successfully!")

Updated hospital_final_dataset.xlsx saved successfully!


In [17]:
import pandas as pd
import os

BASE_DIR = os.path.dirname(os.getcwd())

# Read final dataset
df = pd.read_excel(
    os.path.join(BASE_DIR, "data", "processed", "hospital_final_dataset.xlsx")
)

# KPI Summary
kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Admissions",
        "Occupancy Rate",
        "Average Length of Stay",
        "Readmission Rate",
        "Bed Utilization Rate",
        "Department Efficiency Score"
    ],
    "Value": [
        int(df["Total_Admissions"].iloc[0]),
        df["Occupancy_Rate"].iloc[0],
        df["Average_Length_of_Stay"].iloc[0],
        df["Readmission_Rate"].iloc[0],
        df["Bed_Utilization_Rate"].iloc[0],
        df["Department_Efficiency_Score"].iloc[0]
    ]
})

# Create KPI_Data folder
output_folder = os.path.join(BASE_DIR, "KPI_Data")
os.makedirs(output_folder, exist_ok=True)

# Save CSV
kpi_summary.to_csv(
    os.path.join(output_folder, "KPI_Summary.csv"),
    index=False
)

print("✅ KPI_Summary.csv created successfully!")

✅ KPI_Summary.csv created successfully!


In [19]:
import pandas as pd
import os

# ===============================
# Project Paths
# ===============================
BASE_DIR = os.path.dirname(os.getcwd())

input_file = os.path.join(
    BASE_DIR,
    "data",
    "processed",
    "hospital_final_dataset.xlsx"
)

output_folder = os.path.join(BASE_DIR, "KPI_Data")
os.makedirs(output_folder, exist_ok=True)

# ===============================
# Load Dataset
# ===============================
df = pd.read_excel(input_file)

# ===============================
# 1. KPI Summary
# ===============================
kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Admissions",
        "Occupancy Rate",
        "Average Length of Stay",
        "Readmission Rate",
        "Bed Utilization Rate",
        "Department Efficiency Score"
    ],
    "Value": [
        int(df["Total_Admissions"].iloc[0]),
        float(df["Occupancy_Rate"].iloc[0]),
        float(df["Average_Length_of_Stay"].iloc[0]),
        float(df["Readmission_Rate"].iloc[0]),
        float(df["Bed_Utilization_Rate"].iloc[0]),
        float(df["Department_Efficiency_Score"].iloc[0])
    ]
})

kpi_summary.to_csv(
    os.path.join(output_folder, "KPI_Summary.csv"),
    index=False
)

# ===============================
# 2. Admissions by Month
# ===============================
monthly = (
    df.groupby("admission_year_month")["admission_id"]
      .count()
      .reset_index(name="Admissions")
)

monthly.to_csv(
    os.path.join(output_folder, "Admissions_by_Month.csv"),
    index=False
)

# ===============================
# 3. Department Admissions
# ===============================
department = (
    df.groupby("department_name")["admission_id"]
      .count()
      .reset_index(name="Admissions")
)

department.to_csv(
    os.path.join(output_folder, "Department_Admissions.csv"),
    index=False
)

# ===============================
# 4. Bed Utilization
# ===============================
bed = (
    df.groupby("ward_name")["bed_occupancy_flag"]
      .sum()
      .reset_index(name="Occupied_Beds")
)

bed.to_csv(
    os.path.join(output_folder, "Bed_Utilization.csv"),
    index=False
)

# ===============================
# 5. Readmission Summary
# ===============================
readmission = pd.DataFrame({
    "Readmission_Rate": [df["Readmission_Rate"].iloc[0]]
})

readmission.to_csv(
    os.path.join(output_folder, "Readmission_Summary.csv"),
    index=False
)

# ===============================
# 6. Department Efficiency
# ===============================
efficiency = (
    df.groupby("department_name")["Department_Efficiency_Score"]
      .mean()
      .reset_index()
)

efficiency.to_csv(
    os.path.join(output_folder, "Department_Efficiency.csv"),
    index=False
)

# ===============================
# 7. Average Length of Stay
# ===============================
los = (
    df.groupby("department_name")["length_of_stay"]
      .mean()
      .reset_index(name="Average_Length_of_Stay")
)

los.to_csv(
    os.path.join(output_folder, "Length_of_Stay.csv"),
    index=False
)

# ===============================
# 8. Occupancy Rate
# ===============================
occupancy = pd.DataFrame({
    "Occupancy_Rate": [df["Occupancy_Rate"].iloc[0]]
})

occupancy.to_csv(
    os.path.join(output_folder, "Occupancy_Rate.csv"),
    index=False
)

# ===============================
# Done
# ===============================
print("=" * 50)
print("KPI Data Generated Successfully!")
print("=" * 50)

print("\nFiles Created:")

for file in sorted(os.listdir(output_folder)):
    print(file)

KPI Data Generated Successfully!

Files Created:
Admissions_by_Month.csv
Bed_Utilization.csv
Department_Admissions.csv
Department_Efficiency.csv
KPI_Summary.csv
Length_of_Stay.csv
Occupancy_Rate.csv
Readmission_Summary.csv


In [20]:
print(df.columns.tolist())

['admission_id', 'admission_date', 'discharge_date', 'admission_type', 'admission_status', 'patient_id', 'department_id', 'ward_id', 'bed_id', 'disease_id', 'gender', 'date_of_birth', 'blood_group', 'city', 'contact_number', 'department_name', 'department_type', 'floor_number', 'status', 'ward_name', 'ward_type', 'total_beds', 'bed_number', 'bed_status', 'disease_name', 'disease_category', 'bill_id', 'bill_date', 'total_amount', 'insurance_covered_amount', 'patient_payable_amount', 'payment_status', 'payment_mode', 'patient_insurance_id', 'policy_number', 'coverage_percentage', 'policy_start_date', 'policy_end_date', 'insurance_provider_id', 'provider_name', 'provider_type', 'contact_details', 'coverage_limit', 'length_of_stay', 'patient_age', 'age_group', 'insurance_status', 'revenue_category', 'admission_year', 'admission_month', 'admission_month_number', 'admission_year_month', 'admission_quarter', 'emergency_flag', 'stay_category', 'department_revenue', 'disease_load', 'bed_occupan